# Clean Sales Data — Contoso Retail (Fabric / PySpark)

This notebook is the Fabric equivalent of `clean_sales_data_local.py`.
Same logic, same mock dataset — just expressed as distributed DataFrame
operations instead of plain Python loops.

**Before running:** upload `sales_raw.json`, `products.json`, and
`customers.json` into this Lakehouse's `Files/raw/` folder, and attach
this notebook to your `SalesLakehouse` Lakehouse (View -> Lakehouse explorer
-> Add).

## Step 1 — Read the raw JSON files from Lakehouse Files

In [ ]:
# spark is already available in a Fabric notebook -- no need to create a
# SparkSession yourself. spark.read.json() infers a schema automatically,
# but for messy source data it's often safer to read everything as string
# first and cast explicitly later (see Step 4) rather than trust inference.

sales_raw = spark.read.json("Files/raw/sales_raw.json")
products  = spark.read.json("Files/raw/products.json")
customers = spark.read.json("Files/raw/customers.json")

sales_raw.show(truncate=False)
sales_raw.printSchema()

## Step 2 — Deduplicate: keep the latest row per SalesID

Two issues live in the raw data:
- `SalesID 1005` appears twice as an **exact duplicate**
- `SalesID 1002` appears twice as a **correction** (Amount changed, later `LoadTimestamp`)

The same rule fixes both: for each `SalesID`, keep only the row with the
newest `LoadTimestamp`. This is a classic **window function** pattern --
the Spark-native way to do "keep latest per key" without collecting data
back to the driver.

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc

# partitionBy("SalesID") groups rows by key, orderBy(desc(...)) ranks each
# group's rows newest-first, row_number() == 1 picks the winner. This runs
# in parallel across partitions -- no single node has to see the whole table.
window_spec = Window.partitionBy("SalesID").orderBy(desc("LoadTimestamp"))

deduped = (
    sales_raw
    .withColumn("_rn", row_number().over(window_spec))
    .filter("_rn = 1")
    .drop("_rn")
)

print(f"Raw rows: {sales_raw.count()}  ->  Deduplicated rows: {deduped.count()}")

## Step 3 — Standardize dates, casing, and types

- `Date` mixes two formats: `2026/07/01` and `07-02-2026`. `coalesce()`
  tries each `to_date()` pattern in order and keeps the first non-null result.
- `Region` has inconsistent casing (`west`, `East`, `south`) -- `initcap()` fixes it.
- `Amount` is a string on one row (`"24.99"`) instead of a number --
  `cast("double")` normalizes the type. A value that's already numeric
  just passes through unchanged.
- `ProductID` is genuinely missing on one row. We do **not** drop the row
  or invent a value -- it stays NULL and gets flagged separately below.

In [ ]:
from pyspark.sql.functions import coalesce, to_date, initcap, col, when

cleaned = (
    deduped
    .withColumn(
        "SaleDate",
        coalesce(
            to_date("Date", "yyyy/MM/dd"),
            to_date("Date", "MM-dd-yyyy"),
        ),
    )
    .withColumn("Region", initcap("Region"))
    .withColumn("Amount", col("Amount").cast("double"))
    .select("SalesID", "SaleDate", "ProductID", "CustomerID", "Region", "Amount")
)

cleaned.show(truncate=False)

missing_product = cleaned.filter(col("ProductID").isNull())
print(f"Rows with missing ProductID (flagged, not dropped): "
      f"{[r.SalesID for r in missing_product.collect()]}")

## Step 4 — Detect late-arriving data

Compare the business date (`SaleDate`) to when the row actually arrived
(`LoadTimestamp`). A gap of more than a day means the row is late-arriving --
it must still be filed under its correct `SaleDate`, not the day it happened
to show up, or historical reports would silently shift.

In [ ]:
from pyspark.sql.functions import to_date as to_date_ts, datediff

late_arriving = (
    deduped
    .withColumn("SaleDate2", coalesce(to_date("Date", "yyyy/MM/dd"), to_date("Date", "MM-dd-yyyy")))
    .withColumn("LoadDate", to_date_ts("LoadTimestamp"))
    .withColumn("GapDays", datediff("LoadDate", "SaleDate2"))
    .filter("GapDays > 1")
    .select("SalesID", "SaleDate2", "LoadDate", "GapDays")
)

late_arriving.show()

## Step 5 — Write the cleaned data as a managed Delta table

This is the moment the data actually lands in OneLake as a proper table --
readable afterward by the Warehouse SQL endpoint, Power BI Direct Lake,
or any other Spark notebook, with zero further copying.

In [ ]:
cleaned.write.format("delta").mode("overwrite").saveAsTable("Sales")
products.write.format("delta").mode("overwrite").saveAsTable("Products")
customers.write.format("delta").mode("overwrite").saveAsTable("Customers")

print("Sales, Products, and Customers tables written to the Lakehouse.")

## Step 6 — Sanity check: reproduce the Warehouse aggregation here too

Same question as the T-SQL version from Session 1 -- "sales by region" --
just expressed as a DataFrame `groupBy` instead of `GROUP BY`. Getting the
same shape of answer two different ways is a good way to confirm the
cleaning logic actually did what you think it did.

In [ ]:
from pyspark.sql.functions import sum as _sum, count as _count

result = (
    cleaned
    .join(customers, "CustomerID")
    .groupBy(customers["Region"])
    .agg(
        _sum("Amount").alias("TotalSales"),
        _count("SalesID").alias("OrderCount"),
    )
    .orderBy(desc("TotalSales"))
)

result.show()